# Exercise 2 — Making an LLM API Call using **Groq** (Free API Key)

Dear Professors,  
In this exercise you will make a **single** LLM API call using **Groq**, similar to the OpenAI exercise, but using Groq’s free API key.

## Learning outcomes
By the end of this notebook, you will be able to:
1. Create and use a Groq API key safely (without hard‑coding it).
2. Call a Groq hosted LLM (e.g., Llama) with a short system instruction and a user question.
3. Use a local text file (`students_details.txt`) as *context* for Q&A.

> **Note:** We keep the code intentionally minimal. We will discuss limitations and advanced workflows (LangChain, LangSmith, RAG, Agents, MCP, A2A) in later sessions.


## 0) One-time setup (install the Groq Python library)

If you are running locally or on a fresh environment, install the Groq SDK:

```bash
pip install -U groq
```

If the install is already done, you can skip this step.


In [ ]:
# If you need installation inside the notebook, uncomment the next line:
# !pip install -U groq

## 1) Set your Groq API key (recommended: environment variable)

Please create a Groq API key from the Groq Console (you will be shown this during the workshop).

**Secure approach:** do not paste keys into code cells that you plan to share.  
Instead, we will either:
- read it from your environment variable `GROQ_API_KEY`, or
- prompt you privately (input hidden).



In [ ]:
import os
import getpass

# Preferred: use an environment variable
if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"].strip():
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY (input hidden): ")

print("GROQ_API_KEY is set ✅")

## 2) Load the dataset (`students_details.txt`)

We will load the same dataset format as in the OpenAI exercise.


In [ ]:
from pathlib import Path

DATA_PATH = Path("students_details.txt")

# If you are running this notebook elsewhere, update the path accordingly.
assert DATA_PATH.exists(), f"Could not find {DATA_PATH.resolve()}"

text_data = DATA_PATH.read_text(encoding="utf-8")

print("Loaded characters:", len(text_data))
print("\n--- Preview (first 600 chars) ---\n")
print(text_data[:600])

## 3) A minimal Groq API call (Chat Completions)

We will:
- send the full text as **context**
- ask one question
- print the model’s answer

### Model choice
For classroom demos, a smaller/faster model is usually best. For example:
- `llama-3.1-8b-instant` (fast, good for Q&A)
- `llama3-70b-8192` (older naming, still widely referenced)

If a model name fails, it simply means it is not enabled in the plan or has been renamed; we will provide the latest list in class.


In [ ]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL_ID = "llama-3.1-8b-instant"   # you may change this if instructed in class

SYSTEM_RULES = """
You are a helpful assistant.
Answer ONLY using the provided CONTEXT.
If the answer is not explicitly in the context, say: "Not found in the provided data."
Keep answers short and direct.
"""

def ask_question(question: str, context: str) -> str:
    completion = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"}
        ],
        temperature=0
    )
    return completion.choices[0].message.content

# Quick test
print(ask_question("List the student names in the data.", text_data))

## 4) Interactive Q&A loop (optional)

Type questions one by one.  
Type `exit` to stop.


In [ ]:
while True:
    q = input("\nAsk a question (or type 'exit'): ").strip()
    if q.lower() == "exit":
        print("Exiting...")
        break
    ans = ask_question(q, text_data)
    print("\nAnswer:\n", ans)

## 5) Suggested questions (optional)

- “List all student names.”  
- “Summarize the project of \<student name\>.”  
- “Which students belong to \<class / cohort\>?”  
- “Which project mentions \<keyword\>?”  
